# Cuaderno de Desarrollo — Asistente de Soporte RAG

**Proyecto**: Prueba Técnica — Asistente de Soporte Automatizado (Unilink)
**Autor**: Franco Lionti
**Fecha**: Junio 2026

---

Este cuaderno documenta el proceso de desarrollo, las decisiones técnicas fundamentadas y los conceptos teóricos detrás del sistema RAG implementado. Está diseñado como referencia para la entrevista técnica y como documentación viva del proyecto.

**Contenido:**
1. Resumen del proyecto y arquitectura
2. Evolución y fundamentos de RAG
3. Búsqueda Léxica vs. Vectorial vs. Híbrida
4. Modelos de Embeddings: decisión y justificación
5. Métricas de evaluación de RAG (RAG Triad)
6. Estrategia de optimización de costos
7. RAG vs. Agentes de Comando
8. Observabilidad y métricas light
9. Componentes clave del sistema
10. Benchmarking práctico
12. Ejecución End-to-End Real y medición
11. Decisiones arquitectónicas y tradeoffs


## 1. Resumen del proyecto y arquitectura

### Objetivo
Construir un **asistente de soporte técnico** que responda preguntas de usuarios usando documentación interna, implementado como un sistema **RAG (Retrieval-Augmented Generation)**.

### Flujo del sistema
```
Usuario → n8n (webhook) → FastAPI (/ask) → Embedding de la pregunta
                                         → Búsqueda semántica en ChromaDB
                                         → Contexto relevante → LLM (GPT-4o-mini)
                                         → Respuesta fundamentada → Usuario
```

### Stack tecnológico

| Componente | Tecnología | Justificación |
|---|---|---|
| Procesamiento | Python 3.11+ | Requerido por el enunciado |
| API REST | FastAPI | Async, validación Pydantic, Swagger automático |
| Embeddings | `all-MiniLM-L6-v2` (local) | Sin costo, sin latencia de red, suficiente para docs técnicos |
| Vector Store | ChromaDB | Embebido, sin infraestructura, persistencia en disco |
| LLM | GPT-4o-mini (OpenAI) | Capa gratuita, buena relación costo/calidad |
| Orquestación | n8n | Requerido por el enunciado |
| Deployment | Docker Compose | Todo se levanta con un comando |

### Estructura del proyecto
```
Unilink/
├── src/
│   ├── ingestion/    # Pipeline: readers → normalizer → chunker → embedder → vector_store
│   ├── api/          # FastAPI: endpoints + schemas + métricas
│   └── llm/          # Generator + prompts (OpenAI)
├── tests/            # pytest: unitarios + integración (55 tests)
├── docs/             # Documentación a indexar
├── n8n/              # Workflow exportado
├── Dockerfile + docker-compose.yml
└── development_notebook.ipynb  # ← Este archivo
```


## 2. Evolución y fundamentos de RAG

### ¿De dónde nacen los sistemas RAG?

Los LLMs tienen tres limitaciones fundamentales que RAG resuelve:

#### 1. Ventana de contexto acotada
Los modelos tempranos (GPT-3, 2020) tenían ventanas de 2048-4096 tokens. Incluso los modelos modernos (GPT-4o con 128K tokens) tienen un límite práctico: meter toda la documentación en el contexto es **costoso** (más tokens = más dinero) e **ineficiente** (el modelo pierde rendimiento con contextos muy largos — el efecto "Lost in the Middle" documentado por Liu et al., 2023).

> **RAG resuelve esto**: En lugar de enviar toda la documentación, enviamos solo los 3-5 fragmentos más relevantes. En nuestro caso, enviamos ~1500 caracteres de contexto en lugar de ~15000 del corpus completo.

#### 2. Control de alucinaciones
Un LLM sin contexto externo "inventa" información basándose en sus datos de entrenamiento. Con RAG, el modelo tiene instrucciones explícitas de responder **solo** con lo que encuentra en el contexto proporcionado.

> En nuestro sistema, el `SYSTEM_PROMPT` en `src/llm/prompts.py` dice: *"Si la información no está en el contexto proporcionado, indicá claramente que no tenés esa información."*

#### 3. Auditabilidad y citas de fuentes
Sin RAG, el usuario no sabe de dónde viene la información. Con RAG, cada respuesta incluye las **fuentes** (archivo y fragmento específico) con un **score de relevancia**, permitiendo al usuario verificar la información.

> Nuestro endpoint `/ask` devuelve un array `sources` con `source_file`, `text` y `relevance` para cada chunk usado.

### RAG vs. Fine-tuning

| Aspecto | RAG | Fine-tuning |
|---|---|---|
| Actualización de datos | Instantánea (re-indexar) | Requiere re-entrenar |
| Costo | Bajo (solo inferencia) | Alto (GPU para entrenamiento) |
| Alucinaciones | Controladas (grounding) | Persisten |
| Trazabilidad | Fuentes citables | Caja negra |
| Datos necesarios | Documentos crudos | Miles de ejemplos etiquetados |

**Decisión**: RAG es claramente superior para este caso de uso (documentación que cambia, pocos documentos, necesidad de trazabilidad).


## 3. Búsqueda Léxica vs. Vectorial vs. Híbrida

### Búsqueda Léxica (Keywords)
Basada en coincidencia exacta de términos. Algoritmos como TF-IDF o BM25.

**Fortalezas**:
- Excelente para términos técnicos exactos ("error 502", "timeout TCP")
- Rápida, sin dependencia de modelos
- Determinista y explicable

**Debilidades**:
- No entiende sinónimos: "reiniciar servicio" ≠ "resetear aplicación"
- No captura intención: "no puedo conectarme" no matchea con "error de conexión"

### Búsqueda Vectorial (Semántica)
Convierte texto a vectores densos (embeddings) y busca por similitud coseno.

**Fortalezas**:
- Entiende semántica: "no puedo entrar" matchea con "error de autenticación"
- Multilingüe: embeddings modernos alinean idiomas
- Robusto a variaciones de redacción

**Debilidades**:
- No es buena con IDs exactos: "ticket #12345" o "error 0x80070005"
- Requiere modelo de embeddings (costo computacional)
- Puede traer falsos positivos semánticamente cercanos pero irrelevantes

### Búsqueda Híbrida (RRF — Reciprocal Rank Fusion)
Combina ambos rankings. Para cada documento `d` en posición `rank_i` en el ranking `i`:

$$RRF(d) = \sum_{i=1}^{n} \frac{1}{k + rank_i(d)}$$

donde `k` es una constante (típicamente 60) que suaviza el impacto de las posiciones altas.

**Ejemplo práctico**:
- Un chunk está en posición 1 en BM25 (matchea exacto "error 502") pero en posición 15 en vectorial.
- Otro chunk está en posición 10 en BM25 pero en posición 1 en vectorial (semánticamente relevante).
- RRF los combina para dar un ranking balanceado.

### Nuestra decisión: Vectorial pura (por ahora)

| Factor | Nuestra elección |
|---|---|
| Corpus | 4 documentos, ~19 chunks. Muy pequeño. |
| Tipo de consultas | Preguntas en lenguaje natural ("¿cómo soluciono...?") |
| Precisión necesaria | Alta, pero no crítica (soporte interno, no médico) |

Con un corpus tan pequeño, la búsqueda vectorial con `top_k=3` ya cubre una fracción significativa del total. La complejidad adicional de híbrida no se justifica aún.

> **Regla de evolución**: Si el corpus crece a 100+ documentos y aparecen consultas con IDs/códigos específicos, evaluar BM25 + vectorial con RRF.


## 4. Modelos de Embeddings: decisión y justificación

### Comparativa de opciones evaluadas

| Modelo | Dimensiones | Tamaño | Latencia | Costo | Calidad |
|---|---|---|---|---|---|
| `all-MiniLM-L6-v2` (local) | 384 | ~80MB | ~5ms/query | $0 | Buena |
| `text-embedding-3-small` (OpenAI) | 1536 | N/A (API) | ~100ms/query + red | $0.02/1M tokens | Muy buena |
| `text-embedding-3-large` (OpenAI) | 3072 | N/A (API) | ~150ms/query + red | $0.13/1M tokens | Excelente |

### ¿Por qué elegimos `all-MiniLM-L6-v2`?

1. **Costo cero**: No consume tokens de OpenAI. En un proyecto donde el presupuesto es limitado (tier gratuito), reservar los tokens para la generación de respuestas (GPT-4o-mini) es la decisión correcta.

2. **Sin latencia de red**: Los embeddings se generan localmente en ~5ms. Con OpenAI, cada embedding requiere una llamada HTTP (~100ms + latencia de red). En el flujo RAG, esto impacta directamente la experiencia del usuario.

3. **Sin dependencia externa para la búsqueda**: Si la API de OpenAI se cae, nuestro sistema sigue pudiendo buscar en la documentación. Solo la generación de respuestas depende de OpenAI.

4. **Suficiente para el caso de uso**: Con 19 chunks de documentación técnica en español, 384 dimensiones capturan suficiente semántica. La diferencia de calidad entre 384 y 1536 dims se nota más con corpus grandes y consultas ambiguas.

### Impacto en almacenamiento (ChromaDB)

```
384 dims × 4 bytes/float × 19 chunks = ~29 KB
1536 dims × 4 bytes/float × 19 chunks = ~117 KB
```

Con 19 chunks la diferencia es insignificante. Con 100K chunks:
```
384 dims × 4 bytes × 100K = ~147 MB
1536 dims × 4 bytes × 100K = ~586 MB
```

### Trade-off explícito
Sacrificamos ~10-15% de calidad semántica en embeddings a cambio de:
- Eliminación total de costos de embedding
- Eliminación de latencia de red
- Mayor resiliencia (no depende de un servicio externo)
- Privacidad (los documentos nunca salen del servidor)


## 5. Métricas de Evaluación de RAG — La "RAG Triad"

Para evaluar un sistema RAG de forma rigurosa, se usan tres métricas complementarias conocidas como la **RAG Triad** (popularizada por TruLens y el equipo de Stanford):

### 1. Context Relevance (Relevancia del Contexto)
> *¿Los chunks recuperados son relevantes para la pregunta?*

- **Qué mide**: La calidad del retriever. Si trae chunks irrelevantes, el LLM tiene "ruido" en su contexto.
- **Cómo se mide**: Se le pide a un LLM evaluador que puntúe cada chunk del 0-1 según su relevancia para la pregunta.
- **Nuestro proxy**: El `relevance` score en cada `SourceChunk` (1 - distancia coseno). No es evaluado por LLM, pero da una señal útil.

**Ejemplo en nuestro sistema**:
```json
{
  "sources": [
    {"source_file": "docs/Documentación 2.txt", "relevance": 0.75},
    {"source_file": "docs/Documentación 3.md", "relevance": 0.62}
  ]
}
```
Si `relevance < 0.5`, el chunk probablemente es ruido.

### 2. Groundedness / Faithfulness (Fidelidad)
> *¿La respuesta está fundamentada en el contexto proporcionado?*

- **Qué mide**: Si el LLM "inventa" información o se basa en sus datos de entrenamiento en lugar del contexto.
- **Cómo se mide**: Se descompone la respuesta en afirmaciones (claims) y se verifica si cada una tiene soporte en el contexto.
- **Nuestra mitigación**: El `SYSTEM_PROMPT` instruye al modelo a responder *solo* con información del contexto y a indicar explícitamente cuando no tiene información.

### 3. Answer Relevance (Relevancia de la Respuesta)
> *¿La respuesta contesta la pregunta del usuario?*

- **Qué mide**: Si la respuesta es útil y pertinente. Un sistema puede recuperar buen contexto y ser fiel, pero responder algo que no es lo que el usuario preguntó.
- **Cómo se mide**: Se le pide a un LLM evaluador que genere preguntas a partir de la respuesta, y se mide la similitud coseno con la pregunta original.

### Implementación práctica (próximos pasos)
Para implementar evaluación formal, se necesitaría:
1. Un dataset de evaluación: ~20-50 pares (pregunta, respuesta esperada)
2. Un pipeline de evaluación con LLM-as-judge (GPT-4o evaluando GPT-4o-mini)
3. Métricas automatizadas usando frameworks como **Ragas** o **TruLens**

> **Nota**: En nuestro sistema actual, las métricas de observabilidad en `/ask` (`metrics.top_relevance`, `metrics.chunks_retrieved`) proveen señales proxy útiles sin el costo de un LLM evaluador.


## 6. Estrategia de optimización de costos

### Principio: "Start Big, Optimize Down"

La estrategia recomendada en producción de LLMs es **empezar con los modelos más potentes** para validar la viabilidad del caso de uso, y luego **migrar componentes específicos a modelos más económicos** midiendo el impacto.

### Cascada de modelos (aplicada a nuestro proyecto)

```
Fase 1 (Validación):    GPT-4o         → ¿El RAG puede responder correctamente?
Fase 2 (Optimización):  GPT-4o-mini    → ¿La calidad se mantiene con un modelo 10x más barato?
Fase 3 (Producción):    Modelo local   → ¿Podemos eliminar el costo por completo?
```

### Costos reales de nuestro sistema (por request a `/ask`)

| Componente | Modelo | Costo estimado |
|---|---|---|
| Embedding de la pregunta | all-MiniLM-L6-v2 (local) | **$0** |
| Búsqueda en ChromaDB | Local | **$0** |
| Generación de respuesta | GPT-4o-mini | ~$0.0003/request* |

*Estimado: ~300 tokens prompt + ~200 tokens respuesta a precios de junio 2026.

### Optimizaciones ya implementadas

1. **Embeddings locales**: Ahorro de ~$0.00002/request vs OpenAI embeddings. Parece poco, pero a 10K requests/día = $6/mes ahorrados solo en embeddings.

2. **GPT-4o-mini en lugar de GPT-4o**: ~10x más barato con calidad suficiente para soporte técnico. En nuestras pruebas, las respuestas son igual de precisas para documentación técnica estructurada.

3. **Chunking optimizado (500 chars, overlap 50)**: Menos chunks = menos tokens en el contexto = menos costo por request.

### Próxima optimización: cache de respuestas
Para preguntas frecuentes (>30% del tráfico en soporte), un cache semántico basado en similitud de embeddings de la pregunta evitaría llamadas al LLM por completo:

```python
# Pseudocódigo de cache semántico
cached_embedding = cache.search(query_embedding, threshold=0.95)
if cached_embedding:
    return cached_response  # $0, ~5ms
else:
    response = llm.generate(...)  # ~$0.0003, ~1500ms
    cache.store(query_embedding, response)
```


## 7. RAG vs. Agentes de Comando

### La tendencia actual: Agentes que ejecutan comandos

En los últimos meses (2025-2026), ha surgido una tendencia donde los **agentes de IA** reemplazan o complementan la indexación estática de RAG ejecutando comandos directamente sobre el sistema de archivos:

```
RAG tradicional:  Docs → Indexar → Buscar vectores → Contexto → LLM
Agente de comando: LLM → "necesito buscar X" → grep/find/cat → Contexto → LLM
```

### Comparativa

| Aspecto | RAG (nuestro sistema) | Agente de Comando |
|---|---|---|
| Latencia | Baja (~50ms búsqueda) | Variable (depende del comando) |
| Precisión | Semántica (puede traer ruido) | Exacta (grep es determinista) |
| Actualización | Requiere re-indexar | Siempre actualizado (lee el FS) |
| Costo fijo | Embeddings + almacenamiento | Solo tokens del LLM |
| Complejidad | Pipeline de ingesta | Orquestación de agentes |
| Seguridad | Aislado (solo lee índice) | Riesgo (ejecuta comandos) |
| Escalabilidad | O(1) búsqueda con índice | O(n) búsqueda sin índice |

### ¿Cuándo usar cada uno?

- **RAG**: Corpus estable, necesidad de baja latencia, búsqueda semántica. Nuestro caso ✓
- **Agentes**: Codebase en constante cambio, necesidad de precisión exacta, pocas consultas por día.
- **Híbrido**: El agente decide si buscar en el índice vectorial o ejecutar un comando según la pregunta.

### Nuestro posicionamiento
Implementamos RAG puro porque:
1. La documentación es relativamente estática (no cambia cada hora)
2. Las consultas son semánticas ("¿cómo soluciono...?"), no de búsqueda exacta
3. El costo de indexación es marginal (4 documentos, 19 chunks)
4. La seguridad es importante (no queremos que un LLM ejecute comandos en producción)


## 8. Observabilidad y métricas light

### ¿Qué instrumentamos en `/ask`?

El módulo `src/api/metrics.py` implementa un sistema de telemetría liviano que registra métricas por cada request:

```python
@dataclass
class RequestMetrics:
    # Latencias (ms)
    embedding_latency_ms: float   # Tiempo de generar embedding de la pregunta
    retrieval_latency_ms: float   # Tiempo de búsqueda en ChromaDB
    llm_latency_ms: float         # Tiempo de llamada a OpenAI
    total_latency_ms: float       # Suma de las tres etapas

    # Tamaños (proxy de tokens)
    question_chars: int           # Caracteres de la pregunta
    context_chars: int            # Caracteres del contexto enviado al LLM
    answer_chars: int             # Caracteres de la respuesta

    # Retrieval
    chunks_retrieved: int         # Cuántos chunks se encontraron
    top_relevance: float          # Score del chunk más relevante

    # LLM (tokens reales de la API de OpenAI)
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int
```

### ¿Por qué estas métricas?

1. **Latencia por etapa**: Permite identificar cuellos de botella. Si `llm_latency_ms` >> `retrieval_latency_ms`, el bottleneck está en OpenAI, no en nuestra búsqueda.

2. **Caracteres como proxy de tokens**: Los caracteres son un proxy útil (~4 chars/token en inglés, ~3 en español). Para estimación rápida de costos sin parsear la respuesta del proveedor.

3. **Top relevance**: Si consistentemente es < 0.5, la documentación no cubre las preguntas de los usuarios → señal para expandir el corpus.

### Conexión con plataformas de observabilidad en producción

En un entorno productivo, estas métricas se conectarían a plataformas especializadas:

#### Langfuse (recomendado para LLMOps)
```python
from langfuse import Langfuse
from langfuse.decorators import observe

langfuse = Langfuse()

@observe()  # Decorador que tracéa automáticamente
async def ask_question(body: QuestionRequest):
    # El decorador captura inputs, outputs, latencias y tokens
    # Todo se envía a la dashboard de Langfuse
    ...
```

#### Beneficios en producción
- **Dashboards en tiempo real**: Latencia P50/P95/P99 por etapa
- **Alertas**: Si `llm_latency_ms > 5000` o `top_relevance < 0.3`
- **Análisis de costos**: Gasto diario/mensual por tokens
- **Debugging**: Replay de requests para diagnosticar respuestas incorrectas
- **A/B testing**: Comparar modelos/prompts/chunk_sizes con datos reales


## 9. Componentes clave del sistema

### `src/ingestion/readers.py` — Patrón Strategy
```python
class DocumentReader(ABC):
    @abstractmethod
    def read(self, path: Path) -> Document: ...

class TxtReader(DocumentReader): ...
class MarkdownReader(DocumentReader): ...
class JsonReader(DocumentReader): ...
class PdfReader(DocumentReader): ...  # Bonus

def get_reader(extension: str) -> DocumentReader:  # Factory
```
- **Por qué Strategy**: Agregar un nuevo formato (DOCX, CSV) = crear una clase nueva sin tocar el pipeline.
- **Por qué Factory**: El pipeline no necesita saber qué reader usar; `get_reader(".pdf")` se encarga.

### `src/ingestion/normalizer.py` — Limpieza de texto
- Unicode NFC (caracteres compuestos → canónicos)
- Eliminación de caracteres de control
- Colapso de whitespace
- **Por qué**: Dos textos visualmente idénticos pueden tener embeddings distintos si tienen encoding diferente.

### `src/ingestion/chunker.py` — Chunking semántico
- Split por secciones/párrafos (no corta oraciones)
- Overlap configurable (50 chars por defecto)
- Metadata heredada del documento original
- **Trade-off chunk_size**: 500 chars balancea contexto vs. costo de tokens.

### `src/llm/prompts.py` — Prompt engineering anti-alucinación
- System prompt que fuerza al modelo a usar solo el contexto
- Instrucción explícita de indicar cuando no tiene información
- Instrucción de citar fuentes

### `src/api/metrics.py` — Observabilidad
- `RequestMetrics` dataclass con latencias y tamaños
- `measure()` context manager para timing
- Log estructurado JSON compatible con ELK/Datadog


## 10. Decisiones arquitectónicas y tradeoffs

### 1. Patrón Strategy + Factory para readers
- **Ventaja**: Extensibilidad sin modificar código existente (Open/Closed Principle)
- **Tradeoff**: Ligera sobrecarga estructural, pero gana testabilidad y extensión
- **Validación**: Agregar PdfReader en Fase 7 no requirió cambios en el pipeline

### 2. Normalización antes de embedding
- **Ventaja**: Consistencia: "café" con NFC y NFD produce el mismo embedding
- **Tradeoff**: Ligera pérdida de información (caracteres de control eliminados)
- **Validación**: 11 tests unitarios cubren edge cases

### 3. ChromaDB embebido (no cliente-servidor)
- **Ventaja**: Zero config, persistencia en disco, sin infraestructura
- **Tradeoff**: No escala horizontalmente, single-node
- **Evolución**: Migrar a Pinecone/Weaviate si el corpus supera 100K chunks

### 4. Embeddings locales + LLM remoto
- **Ventaja**: Equilibrio costo/calidad. Embeddings gratis, LLM de calidad
- **Tradeoff**: Dependencia de OpenAI para la generación
- **Mitigación**: El sistema funciona para búsqueda incluso si OpenAI está caído

### 5. Métricas en el response body (no solo en logs)
- **Ventaja**: El frontend/n8n puede mostrar latencias al usuario o tomar decisiones
- **Tradeoff**: Aumenta ligeramente el tamaño de la respuesta
- **Justificación**: La transparencia operativa supera el costo marginal


## 11. Setup y benchmarking práctico

### Setup rápido

```bash
# Clonar y entrar
git clone https://github.com/FrancoLionti/Prueba-Tecnica-Franco-Lionti.git
cd Prueba-Tecnica-Franco-Lionti

# Entorno virtual
python3 -m venv venv
source venv/bin/activate
pip install -r requirements.txt

# Configurar API key
cp .env.example .env
# Editar .env con tu OPENAI_API_KEY

# Ejecutar tests (sin costo de tokens)
pytest

# Levantar la API
uvicorn src.api.main:app --reload
```


In [ ]:
# Carga de documentos y generación de chunks (usando el código del repo)
import os
import time
from pprint import pprint

try:
    from src.config import DOCS_DIR, SUPPORTED_EXTENSIONS
    from src.ingestion.readers import get_reader
    from src.ingestion.chunker import chunk_document
except Exception as e:
    raise RuntimeError(
        "Error importando módulos del repo. Asegurate de ejecutar con PYTHONPATH=. y un entorno virtual activo. Detalle: "
        + str(e)
    )

def load_all_chunks(docs_dir=DOCS_DIR):
    chunks = []
    files_scanned = 0
    for root, _, files in os.walk(docs_dir):
        for f in files:
            ext = os.path.splitext(f)[1].lower()
            if ext in SUPPORTED_EXTENSIONS:
                files_scanned += 1
                path = os.path.join(root, f)
                reader = get_reader(path)
                doc = reader.read(path)
                doc_chunks = chunk_document(doc)
                for c in doc_chunks:
                    chunks.append({
                        'text': c.text,
                        'source': c.source_file,
                        'chunk_index': c.chunk_index,
                        'metadata': getattr(c, 'metadata', {})
                    })
    print(f"Archivos escaneados: {files_scanned}, chunks totales: {len(chunks)}")
    return chunks

chunks = load_all_chunks()
pprint({'sample_chunk_count': len(chunks), 'sample_first': chunks[0] if chunks else None})

In [ ]:
# Benchmark simple: tiempos de encoding, tamaño en disco y latencia de consulta
import os
import time
import random
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

def benchmark_models(models, chunks, sample_queries=100, topk=5, tmp_dir='tmp_bench'):
    os.makedirs(tmp_dir, exist_ok=True)
    texts = [c['text'] for c in chunks]
    results = {}
    for name in models:
        print('\n== Benchmarking', name)
        model = SentenceTransformer(name)
        t0 = time.perf_counter()
        embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
        t1 = time.perf_counter()
        total_time = t1 - t0
        per_item = total_time / max(1, len(texts))
        size_bytes = embeddings.nbytes
        npy_path = os.path.join(tmp_dir, name.replace('/', '_') + '.npy')
        np.save(npy_path, embeddings)
        file_size = os.path.getsize(npy_path + '.npy') if os.path.exists(npy_path + '.npy') else size_bytes

        # build simple brute-force cosine index (NearestNeighbors with brute)
        nn = NearestNeighbors(n_neighbors=topk, metric='cosine', algorithm='brute')
        nn.fit(embeddings)

        # sample queries
        q_count = min(sample_queries, len(texts))
        idxs = random.sample(range(len(texts)), q_count)
        query_times = []
        for i in idxs:
            q = embeddings[i:i+1]
            tq0 = time.perf_counter()
            _ = nn.kneighbors(q, n_neighbors=topk)
            tq1 = time.perf_counter()
            query_times.append(tq1 - tq0)
        avg_query_ms = (sum(query_times) / len(query_times)) * 1000 if query_times else None

        results[name] = {
            'total_chunks': len(texts),
            'encoding_total_s': total_time,
            'encoding_per_item_s': per_item,
            'embeddings_size_bytes': int(size_bytes),
            'saved_file_bytes': int(file_size),
            'avg_query_ms': avg_query_ms,
            'embedding_dim': embeddings.shape[1] if embeddings.ndim == 2 else None
        }
        print('results:', results[name])
    return results

# Example: comparar dos modelos locales
models_to_test = [
    'all-MiniLM-L6-v2',
    'paraphrase-multilingual-MiniLM-L12-v2'
]
if not chunks:
    print('No hay chunks — asegúrate de haber corrido la celda de carga de chunks y que `docs/` contenga archivos soportados.')
else:
    bench_results = benchmark_models(models_to_test, chunks, sample_queries=100, topk=5)
    from pprint import pprint
    pprint(bench_results)

## 13. Interpretación de resultados de benchmarking

### Métricas clave a observar

| Métrica | Qué indica | Acción si es alta |
|---|---|---|
| `encoding_per_item_s` | Latencia de embedding | Considerar modelo más pequeño o GPU |
| `embeddings_size_bytes` | Impacto en almacenamiento | Evaluar dimensiones menores |
| `avg_query_ms` | Experiencia del usuario en retrieval | Optimizar indexado (FAISS, HNSW) |
| `embedding_dim` | Presupuesto de almacenamiento/latencia | Confirma la dimensión elegida |

### Resultados esperados con nuestro setup

Para `all-MiniLM-L6-v2` con 19 chunks:
- **Encoding**: ~5ms/item (CPU, sin GPU)
- **Almacenamiento**: ~29 KB (insignificante)
- **Query latency**: <10ms en ChromaDB local
- **Total /ask latency**: ~1.5-3s (dominado por la llamada a OpenAI)


## 12. Ejecución End-to-End Real y Medición de Métricas

Para validar el sistema completo de forma real (sin mocks), podemos importar la función del endpoint principal de la API (`ask_question`) y ejecutar una consulta real que:
1. Genere el embedding de la pregunta usando el modelo local `all-MiniLM-L6-v2`.
2. Busque los chunks más relevantes en la base de datos local `ChromaDB`.
3. Genere la respuesta en lenguaje natural consultando la API oficial de OpenAI (`gpt-4o-mini`).
4. Extraiga y formatee las métricas de rendimiento por etapa (latencia, caracteres, tokens consumidos).

Ejecutá la celda de abajo para ver la respuesta real y su desglose de telemetría.

In [ ]:
import asyncio
import json
from src.api.main import ask_question
from src.api.schemas import QuestionRequest

# Creamos la petición de prueba con una pregunta técnica típica
request = QuestionRequest(question="¿Cómo soluciono el error de conexión a la base de datos?")

# Dado que Jupyter corre en un event loop activo, podemos usar await directamente
print("1. Ejecutando consulta de warm-up (calentamiento de modelos y carga de Chroma)...")
await ask_question(request)

print("2. Ejecutando consulta end-to-end real con telemetría en caliente...")
response = await ask_question(request)

print("\n=== RESPUESTA DE LA API REAL ===")
print(f"Pregunta: {request.question}")
print(f"Respuesta:\n{response.answer}\n")
print("=== FUENTES UTILIZADAS ===")
for i, src in enumerate(response.sources, 1):
    print(f"[{i}] {src.source_file} (Relevancia: {src.relevance})")

print("\n=== TELEMETRÍA DETALLADA (Métricas) ===")
print(json.dumps(response.metrics, indent=2))

## 14. Conclusiones y preguntas frecuentes de entrevista

### ¿Por qué elegiste ese chunk size?
500 caracteres es un balance empírico: captura una idea completa (error + causa + solución típica) sin desperdiciar tokens. Con documentos técnicos estructurados por secciones, el chunking semántico respeta los límites naturales del texto.

### ¿Qué pasa si la documentación crece a 1000 archivos?
- El embedding local escala linealmente (más tiempo de ingesta, pero es offline)
- ChromaDB soporta millones de vectores
- Considerar búsqueda híbrida (BM25 + vectorial) para mejorar precision
- Evaluar HNSW en lugar de brute-force para queries

### ¿Cómo garantizás que no invente información?
1. System prompt anti-alucinación
2. Contexto limitado a chunks recuperados
3. Score de relevancia expuesto al usuario
4. Métricas de observabilidad para detectar respuestas sin grounding

### ¿Qué métricas de evaluación de RAG implementarías?
La RAG Triad (sección 5): Context Relevance, Groundedness y Answer Relevance. Con un dataset de evaluación de ~50 pares, usando Ragas o TruLens.

### ¿Cómo estructurás la observabilidad?
Métricas light por request (latencias, tokens, relevancia) con logging estructurado JSON. En producción, Langfuse o Datadog para dashboards, alertas y debugging.
